# SpaceX Falcon 9 — Data Wrangling

**Author:** Piyu
**Course:** IBM Data Science Professional Certificate — Applied Data Science Capstone

## Objective
Falcon 9 is the only orbital class rocket that can land its first stage and fly it again, which is the main reason SpaceX launches are far cheaper than competitors (~$62M vs ~$165M+). If we can predict whether the first stage will land successfully, we can estimate the cost of a launch.

In this notebook we take the raw SpaceX launch log and engineer a binary **Class** label:
- `1` → first stage landed successfully
- `0` → first stage did not land (or no landing was attempted)

This label becomes the target variable for the machine learning model later in the project.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv('../data/spacex_raw.csv')
df.head()

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing _Outcome
0,2010-04-06,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-08-12,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-08-10,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-01-03,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


## 1. Initial inspection

In [2]:
print(f"Shape: {df.shape}")
df.info()

Shape: (101, 10)
<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Date               101 non-null    str  
 1   Time (UTC)         101 non-null    str  
 2   Booster_Version    101 non-null    str  
 3   Launch_Site        101 non-null    str  
 4   Payload            101 non-null    str  
 5   PAYLOAD_MASS__KG_  101 non-null    int64
 6   Orbit              101 non-null    str  
 7   Customer           101 non-null    str  
 8   Mission_Outcome    101 non-null    str  
 9   Landing _Outcome   101 non-null    str  
dtypes: int64(1), str(9)
memory usage: 8.0 KB


In [3]:
# Check for missing values per column
df.isnull().sum()

Date                 0
Time (UTC)           0
Booster_Version      0
Launch_Site          0
Payload              0
PAYLOAD_MASS__KG_    0
Orbit                0
Customer             0
Mission_Outcome      0
Landing _Outcome     0
dtype: int64

## 2. Cleaning the Payload Mass column

`PAYLOAD_MASS__KG_` has no missing values, but where it does in practice, the standard approach is to impute with the column mean rather than dropping rows — we don't want to lose launch records for a single missing numeric feature.

In [4]:
df['PAYLOAD_MASS__KG_'] = pd.to_numeric(df['PAYLOAD_MASS__KG_'], errors='coerce')
mean_payload = df['PAYLOAD_MASS__KG_'].mean()
df['PAYLOAD_MASS__KG_'] = df['PAYLOAD_MASS__KG_'].fillna(mean_payload)
print(f"Mean payload mass used for imputation: {mean_payload:.2f} kg")
df['PAYLOAD_MASS__KG_'].isnull().sum()

Mean payload mass used for imputation: 6138.29 kg


np.int64(0)

## 3. Engineering the landing outcome label

The raw `Landing _Outcome` column has free-text values like `'Success (ground pad)'`, `'Failure (drone ship)'`, `'No attempt'`, etc. We collapse these into a binary `Class`:

- Outcomes containing **"Success"** (ground pad or drone ship) → `1`
- Everything else (failure, no attempt, uncontrolled, precluded) → `0`

In [5]:
df['Landing _Outcome'].value_counts()

Landing _Outcome
Success                   38
No attempt                21
Success (drone ship)      14
Success (ground pad)       9
Controlled (ocean)         5
Failure (drone ship)       5
Failure                    3
Failure (parachute)        2
Uncontrolled (ocean)       2
Precluded (drone ship)     1
No attempt                 1
Name: count, dtype: int64

In [6]:
def landing_class(outcome):
    outcome = str(outcome).strip()
    if outcome.startswith('Success'):
        return 1
    return 0

df['Class'] = df['Landing _Outcome'].apply(landing_class)
df[['Landing _Outcome', 'Class']].drop_duplicates().sort_values('Class')

,Landing _Outcome,Class
0,Failure (parachute),0
2,No attempt,0
5,Uncontrolled (ocean),0
8,Controlled (ocean),0
13,Failure (drone ship),0
18,Precluded (drone ship),0
73,No attempt,0
64,Failure,0
22,Success (drone ship),1
19,Success (ground pad),1


In [7]:
success_rate = df['Class'].mean()
print(f"Overall first-stage landing success rate: {success_rate:.2%}")
df['Class'].value_counts()

Overall first-stage landing success rate: 60.40%


Class
1    61
0    40
Name: count, dtype: int64

## 4. Cleaning Booster Version and Launch Site naming

In [8]:
# Booster version strings have inconsistent spacing (e.g. "F9 v1.0  B0003") - normalize whitespace
df['Booster_Version'] = df['Booster_Version'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Launch site names also have minor formatting differences - standardize
df['Launch_Site'] = df['Launch_Site'].str.strip()

df[['Booster_Version', 'Launch_Site']].drop_duplicates().head(10)

,Booster_Version,Launch_Site
0,F9 v1.0 B0003,CCAFS LC-40
1,F9 v1.0 B0004,CCAFS LC-40
2,F9 v1.0 B0005,CCAFS LC-40
3,F9 v1.0 B0006,CCAFS LC-40
4,F9 v1.0 B0007,CCAFS LC-40
5,F9 v1.1 B1003,VAFB SLC-4E
6,F9 v1.1,CCAFS LC-40
11,F9 v1.1 B1011,CCAFS LC-40
12,F9 v1.1 B1010,CCAFS LC-40
13,F9 v1.1 B1012,CCAFS LC-40


## 5. Extracting flight number

We add a `FlightNumber` column (chronological order of launches), since later EDA looks at trends across the launch history.

In [9]:
df = df.sort_values('Date').reset_index(drop=True)
df['FlightNumber'] = range(1, len(df) + 1)
df[['FlightNumber', 'Date', 'Booster_Version', 'Launch_Site', 'Class']].head()

,FlightNumber,Date,Booster_Version,Launch_Site,Class
0,1,2010-04-06,F9 v1.0 B0003,CCAFS LC-40,0
1,2,2010-08-12,F9 v1.0 B0004,CCAFS LC-40,0
2,3,2012-05-22,F9 v1.0 B0005,CCAFS LC-40,0
3,4,2012-08-10,F9 v1.0 B0006,CCAFS LC-40,0
4,5,2013-01-03,F9 v1.0 B0007,CCAFS LC-40,0


## 6. Final wrangled dataset

In [10]:
final_cols = ['FlightNumber', 'Date', 'Booster_Version', 'Launch_Site', 'Payload',
              'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome',
              'Landing _Outcome', 'Class']
df_clean = df[final_cols].rename(columns={'PAYLOAD_MASS__KG_': 'PayloadMass'})
df_clean.head()

,FlightNumber,Date,Booster_Version,Launch_Site,Payload,PayloadMass,Orbit,Customer,Mission_Outcome,Landing _Outcome,Class
0,1,2010-04-06,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute),0
1,2,2010-08-12,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute),0
2,3,2012-05-22,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt,0
3,4,2012-08-10,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt,0
4,5,2013-01-03,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt,0


In [11]:
df_clean.to_csv('../data/dataset_wrangled.csv', index=False)
print(f"Saved wrangled dataset: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")

Saved wrangled dataset: 101 rows, 11 columns


## Summary

- Imputed missing payload mass values with the column mean
- Engineered the binary `Class` target (1 = successful landing, 0 = not) from the free-text landing outcome
- Normalized inconsistent whitespace in booster version and launch site fields
- Found an overall first-stage landing success rate across the dataset (printed above)
- Exported `dataset_wrangled.csv` for use in the EDA and machine learning notebooks